# Width Selection: Red Circle

Comparing circle widths (2, 4, 6, 8) across four levels:
1. Visual clarity
2. CLIP recognition of the marking
3. SpLiCE concept activations per object
4. Aggregation (sum + count) over all objects + full image

Test images:
- Large object: 2007_000032.jpg (aeroplane)
- Small objects: 2007_000033.jpg 

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import torch
import open_clip
import splice
from splice.model import SPLICE
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
import torchvision.transforms.v2 as T
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from utils import draw_box_on_image, draw_circle_on_image

WIDTHS = [2, 4, 6, 8]

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn

VOC_IMG = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/JPEGImages/"
VOC_ANN = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/Annotations/"

to_tensor = T.Compose([T.Resize((640, 640)), T.ToTensor()])

def load_voc_image(filename):
    pil = Image.open(VOC_IMG + filename).convert("RGB")
    tensor = to_tensor(pil)
    return pil, tensor

def load_voc_boxes(filename):
    tree = ET.parse(VOC_ANN + filename.replace(".jpg", ".xml"))
    root = tree.getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        bb = obj.find("bndbox")
        x1, y1 = int(bb.find("xmin").text), int(bb.find("ymin").text)
        x2, y2 = int(bb.find("xmax").text), int(bb.find("ymax").text)
        # scale to 640x640
        orig_w, orig_h = Image.open(VOC_IMG + filename).size
        x1 = int(x1 * 640 / orig_w)
        x2 = int(x2 * 640 / orig_w)
        y1 = int(y1 * 640 / orig_h)
        y2 = int(y2 * 640 / orig_h)
        boxes.append(torch.tensor([x1, y1, x2, y2]))
        labels.append(obj.find("name").text)
    return boxes, labels

test_images = {
    "large_obj":  "2007_000032.jpg",
    "small_objs": "2007_000170.jpg",   # person + bottle — kleine Objekte
}

data = {}
for name, fname in test_images.items():
    pil, tensor = load_voc_image(fname)
    boxes, labels = load_voc_boxes(fname)
    data[name] = {"pil": pil, "tensor": tensor, "boxes": boxes, "labels": labels, "fname": fname}
    print(f"{name}: {len(boxes)} objects: {labels}")


## Step 1: Visual Clarity

For each image and each width: show all detected objects with the circle marking.

In [ ]:
for name, d in data.items():
    boxes = d["boxes"]
    tensor = d["tensor"]
    n_objs = len(boxes)

    fig, axes = plt.subplots(n_objs, len(WIDTHS) + 1, 
                             figsize=(4 * (len(WIDTHS) + 1), 4 * n_objs))
    
    # ensure 2D axes array
    if n_objs == 1:
        axes = axes[None, :]

    for j, box in enumerate(boxes):
        # Original
        axes[j, 0].imshow(d["pil"])
        axes[j, 0].set_title("Original" if j == 0 else "")
        axes[j, 0].axis("off")

        # Each width
        for w_idx, width in enumerate(WIDTHS):
            circle_img = draw_circle_on_image(tensor, box, width=width)
            axes[j, w_idx + 1].imshow(circle_img)
            axes[j, w_idx + 1].set_title(f"width={width}" if j == 0 else "")
            axes[j, w_idx + 1].axis("off")

    plt.suptitle(f"{name} — {d['fname']}", fontsize=13)
    plt.tight_layout()
    plt.show()

## Step 2: CLIP Recognition of the Marking

For each width: compute similarity between the marked image and text prompts.
Does CLIP "see" the red circle better with larger widths?

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_model.eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")
print("CLIP loaded")

In [ ]:
prompts = [
    "a red circle around an aeroplane",
    "a photo of an aeroplane",
    "an aeroplane at an airport",
    "a red circle marking",
]
text_tokens = tokenizer(prompts)

with torch.no_grad():
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

WIDTHS = [2, 4, 6, 8]

d = data["large_obj"]
box = d["boxes"][0]  # Object 1 — large aeroplane

print(f"large_obj — Object 1 (aeroplane) {box.tolist()}")
print(f"{'Prompt':<35} " + " ".join([f"w={w:<6}" for w in WIDTHS]))
print("-" * 70)

scores_per_width = []
for width in WIDTHS:
    circle_pil = draw_circle_on_image(d["tensor"], box, width=width)
    img_t = clip_preprocess(circle_pil).unsqueeze(0)
    with torch.no_grad():
        img_feat = clip_model.encode_image(img_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    scores = (img_feat @ text_feat.T).squeeze(0)
    scores_per_width.append(scores)

for p_idx, prompt in enumerate(prompts):
    row = f"{prompt:<35} "
    row += " ".join([f"{scores_per_width[w_idx][p_idx].item():.3f}  " for w_idx in range(len(WIDTHS))])
    print(row)


In [ ]:
for i, (box, label) in enumerate(zip(data["small_objs"]["boxes"], data["small_objs"]["labels"])):
    print(f"Object {i+1}: {label} — {box.tolist()}")


In [ ]:
prompts = [
    "a red circle around a bottle",
    "a photo of a bottle",
    "a bottle on a table",
    "a red circle marking",
    "a man drinking alcohol",
]
text_tokens = tokenizer(prompts)

with torch.no_grad():
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

d = data["small_objs"]
box = d["boxes"][0]  # Object 1 — small bottle

print(f"small_objs — Object 1 (bottle) {box.tolist()}")
print(f"{'Prompt':<35} " + " ".join([f"w={w:<6}" for w in WIDTHS]))
print("-" * 70)

scores_per_width = []
for width in WIDTHS:
    circle_pil = draw_circle_on_image(d["tensor"], box, width=width)
    img_t = clip_preprocess(circle_pil).unsqueeze(0)
    with torch.no_grad():
        img_feat = clip_model.encode_image(img_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    scores = (img_feat @ text_feat.T).squeeze(0)
    scores_per_width.append(scores)

for p_idx, prompt in enumerate(prompts):
    row = f"{prompt:<35} "
    row += " ".join([f"{scores_per_width[w_idx][p_idx].item():.3f}  " for w_idx in range(len(WIDTHS))])
    print(row)

In [ ]:
box = d["boxes"][2]  # Object 3 — bottle [414, 252, 450, 351]

prompts = [
    "a red circle around a bottle",
    "a photo of a bottle",
    "a bottle on a table",
    "a red circle marking",
]
text_tokens = tokenizer(prompts)

with torch.no_grad():
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

print(f"small_objs — Object 3 (bottle) {box.tolist()}")
print(f"{'Prompt':<35} " + " ".join([f"w={w:<6}" for w in WIDTHS]))
print("-" * 70)

scores_per_width = []
for width in WIDTHS:
    circle_pil = draw_circle_on_image(d["tensor"], box, width=width)
    img_t = clip_preprocess(circle_pil).unsqueeze(0)
    with torch.no_grad():
        img_feat = clip_model.encode_image(img_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    scores = (img_feat @ text_feat.T).squeeze(0)
    scores_per_width.append(scores)

for p_idx, prompt in enumerate(prompts):
    row = f"{prompt:<35} "
    row += " ".join([f"{scores_per_width[w_idx][p_idx].item():.3f}  " for w_idx in range(len(WIDTHS))])
    print(row)

## Step 3: SpLiCE Concept Activations per Object

How do concept activations change with different circle widths?
Comparing top-5 concepts for each width on the same object.

In [ ]:
concepts = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location="cpu")
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location="cpu")
clip_backbone_laion, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone_laion.eval()
splice_model = SPLICE(image_mean=image_mean, dictionary=concepts, clip=clip_backbone_laion, device="cpu", l1_penalty=0.25, return_weights=True)
splice_model.eval()
vocab = splice.get_vocabulary("laion", vocabulary_size=10000)

In [ ]:
def get_splice_emb(image_pil):
    img_t = splice_preprocess(image_pil).unsqueeze(0)
    with torch.no_grad():
        emb, _ = splice_model.encode_image(img_t)
    return emb.squeeze(0)

def top_concepts(emb, vocab, top_k=5):
    idx = emb.argsort(descending=True)[:top_k]
    return [(vocab[i], round(emb[i].item(), 3)) for i in idx]

# Test: large aeroplane (Object 1) + small bottle (Object 3)
test_objects = [
    ("large_obj",  0, "aeroplane"),
    ("small_objs", 2, "bottle"),
]

for img_name, obj_idx, label in test_objects:
    d = data[img_name]
    box = d["boxes"][obj_idx]

    print(f"\n{'='*75}")
    print(f"{img_name} — Object {obj_idx+1} ({label}) {box.tolist()}")
    print(f"{'Concept':<20} " + " ".join([f"w={w:<10}" for w in WIDTHS]))
    print("-" * 75)

    embs = {}
    all_concepts = set()
    for width in WIDTHS:
        circle_pil = draw_circle_on_image(d["tensor"], box, width=width)
        emb = get_splice_emb(circle_pil)
        embs[width] = emb
        for concept, _ in top_concepts(emb, vocab, top_k=5):
            all_concepts.add(concept)

    # Show top-5 per width side by side
    for width in WIDTHS:
        top = top_concepts(embs[width], vocab, top_k=5)
        print(f"\n  w={width}:")
        for concept, val in top:
            print(f"    {concept:<20} {val:.3f}")

## Step 4: Aggregation (Sum + Count) over all objects + full image

How does width affect the final aggregated representation?
Testing sum and count aggregation for large_obj image.

In [ ]:
from utils import obj_enc_aggregation

splice_transform = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711)),
])

d = data["large_obj"]
full_emb = get_splice_emb(d["pil"])

print("Width | Sum: top concept (score) | Count: concepts activated")
print("-" * 65)

for width in WIDTHS:
    obj_embs = torch.stack([
        get_splice_emb(draw_circle_on_image(d["tensor"], box, width=width))
        for box in d["boxes"]
    ])

    # Sum aggregation
    sum_agg = full_emb + obj_embs.sum(dim=0)
    sum_top = top_concepts(sum_agg, vocab, top_k=1)[0]

    # Count aggregation
    full_batch = full_emb.unsqueeze(0)
    obj_batch  = obj_embs.unsqueeze(0)
    count_agg  = obj_enc_aggregation("count", obj_batch, full_batch).squeeze(0)
    n_activated = int((count_agg >= 3).sum().item())

    print(f"w={width:<3} | {sum_top[0]:<20} ({sum_top[1]:.3f}) | {n_activated} concepts activated by ≥3 sources")


In [ ]:
d = data["small_objs"]
full_emb = get_splice_emb(d["pil"])

print("Width | Sum: top concept (score) | Count: concepts activated")
print("-" * 65)

for width in WIDTHS:
    obj_embs = torch.stack([
        get_splice_emb(draw_circle_on_image(d["tensor"], box, width=width))
        for box in d["boxes"]
    ])

    # Sum aggregation
    sum_agg = full_emb + obj_embs.sum(dim=0)
    sum_top = top_concepts(sum_agg, vocab, top_k=1)[0]

    # Count aggregation
    full_batch = full_emb.unsqueeze(0)
    obj_batch  = obj_embs.unsqueeze(0)
    count_agg  = obj_enc_aggregation("count", obj_batch, full_batch).squeeze(0)
    n_activated = int((count_agg >= 3).sum().item())

    print(f"w={width:<3} | {sum_top[0]:<20} ({sum_top[1]:.3f}) | {n_activated} concepts activated by ≥3 sources")


In [ ]:
d = data["small_objs"]
full_emb = get_splice_emb(d["pil"])

# w=4 (best width)
obj_embs = torch.stack([
    get_splice_emb(draw_circle_on_image(d["tensor"], box, width=4))
    for box in d["boxes"]
])

# Sum aggregation
sum_agg  = full_emb + obj_embs.sum(dim=0)
sum_top10 = top_concepts(sum_agg, vocab, top_k=10)

# Count aggregation
full_batch = full_emb.unsqueeze(0)
obj_batch  = obj_embs.unsqueeze(0)
count_agg  = obj_enc_aggregation("count", obj_batch, full_batch).squeeze(0)
count_top10 = top_concepts(count_agg, vocab, top_k=10)

print(f"{'Sum (w=4)':<30} {'Count (w=4)':<30}")
print("-" * 60)
for i in range(10):
    s = f"{sum_top10[i][0]} ({sum_top10[i][1]:.3f})"
    c = f"{count_top10[i][0]} ({int(count_top10[i][1])})"
    print(f"{s:<30} {c:<30}")